In [ ]:
import pandas as pd
import sqlite3

Tables = {}
Tables['Capacitors'] = ['Comment','Voltage','Tolerance','Dielectric','ESR','Footprint']
Tables['Connectors'] = ['Comment','Current','Voltage','Other']
Tables['Integrated Circuits'] = ['Comment','Voltage','Current','Other']
Tables['Resistors'] = ['Comment','Power','Tolerance','Footprint']
Tables['Mechanical'] = ['Comment','Voltage','Current','Other']
Tables['Diodes'] = ['Comment','VF','Current','VR','trr','Other']
type_df = pd.read_csv('TypeList.csv',dtype=str)

# Regenerate Description based on parameters
# Check Category and Type
# Fill in empty cells with '-'
for t in Tables:
    parts = pd.read_csv(f'{t}.csv',dtype=str)
    parts = parts.fillna('-')
    for row in parts.iterrows():
        # Check Single Match of Category and type
        filtered_df = type_df[type_df['Cat'].str.fullmatch(row[1]['Cat']) & type_df['Type'].str.fullmatch(row[1]['Type'])]
        if len(filtered_df) != 1:
            print(f'Table {t} Row {row[0]} {filtered_df} {parts.loc[row[0]]}')
        desc = f'{filtered_df['Cat_Ab'].iloc[0]} {filtered_df['Type_Ab'].iloc[0]}'
        for param in Tables[t]:
            if row[1][param] != '-':
                desc = desc + ' ' + row[1][param]
        parts.loc[row[0],'Description'] = desc
    parts.to_csv(t+'.csv',index=False)


In [2]:
sql_lib = sqlite3.connect('local_lib.sqlite')
for t in Tables:
    df = pd.read_csv(f'{t}.csv',dtype=str)
    df.to_sql(t, sql_lib, if_exists='replace', index=False)
sql_lib.close()